# Postprocessing Readiness

Read-only monitoring of outputs produced by `postprocess_pipeline.py`. A combination is ready for classification only when canonical Stage 1 and Stage 2 IDs match and all derived artifacts are present.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from scripts.data_root import resolve_data_root
from processing_monitoring_utils import scan_postprocessing_readiness

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 300)

DATA_ROOT = resolve_data_root()
PROCESSED_ROOT = DATA_ROOT / "processed"

## Split Readiness

In [ ]:
readiness = scan_postprocessing_readiness(PROCESSED_ROOT)

display(readiness)
display(readiness[readiness["status"] != "READY"])

## Dataset and Model Summary

In [ ]:
summary = (
    readiness.groupby(["dataset", "model"], as_index=False)
    .agg(
        splits=("split", "count"),
        ready_splits=("status", lambda values: (values == "READY").sum()),
        stage1_rows=("stage1_rows", "sum"),
        stage2_rows=("stage2_rows", "sum"),
        missing_stage2=("missing_stage2", "sum"),
        extra_stage2=("extra_stage2", "sum"),
        metrics_rows=("metrics_rows", "sum"),
        similarity_stats_rows=("similarity_stats_rows", "sum"),
        local_evidence_rows=("local_evidence_rows", "sum"),
    )
    .assign(ready=lambda frame: frame["ready_splits"] == frame["splits"])
)
display(summary)

## Readiness Overview

In [ ]:
status_matrix = readiness.pivot_table(index="dataset", columns="model", values="status", aggfunc="first")
status_values = status_matrix.map(lambda value: 1 if value == "READY" else 0)

plt.figure(figsize=(max(8, len(status_matrix.columns) * 2), max(4, len(status_matrix) * 0.6)))
sns.heatmap(status_values, annot=status_matrix, fmt="", cmap=["#f2c14e", "#2a9d8f"], cbar=False)
plt.title("Classification readiness by dataset and model")
plt.xlabel("Model")
plt.ylabel("Dataset")
plt.tight_layout()
plt.show()

processed_texts = readiness.pivot_table(
    index=["dataset", "split"],
    columns="model",
    values="stage1_rows",
    aggfunc="sum",
).sort_index()
processed_texts.index = [f"{dataset} / {split}" for dataset, split in processed_texts.index]

plt.figure(figsize=(max(6, len(processed_texts.columns) * 2), max(4, len(processed_texts) * 0.45)))
sns.heatmap(
    processed_texts,
    annot=processed_texts,
    fmt="g",
    cmap="Blues",
    cbar_kws={"label": "Processed texts"},
    linewidths=0.5,
    linecolor="white",
)
plt.title("Processed text count by dataset/split and model")
plt.xlabel("Model")
plt.ylabel("Dataset / split")
plt.tight_layout()
plt.show()

artifact_columns = ["stage1_rows", "stage2_rows", "metrics_rows", "similarity_stats_rows", "local_evidence_rows"]
counts = readiness.melt(id_vars=["dataset", "model", "split"], value_vars=artifact_columns, var_name="artifact", value_name="rows")
count_plot = sns.catplot(data=counts, kind="bar", x="artifact", y="rows", hue="split", col="model", col_wrap=3, sharey=False, height=3.5, aspect=1.5)
count_plot.set_xticklabels(rotation=45, ha="right")
count_plot.set_axis_labels("Artifact", "Rows")
count_plot.fig.suptitle("Postprocessed artifact row counts", y=1.02)
plt.show()
plt.close(count_plot.fig)